This notebook is a small CPU-friendly boilerplate for exploring LLM inference.

Possible models:
- Qwen/Qwen3-0.6B
- google/gemma-3-1b-it
- meta-llama/Llama-3.2-1B

## 1. Setup

In [ ]:
!pip install -r requirements.txt

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"{DEVICE=}")

## 2. Load Tokenizer And Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print(f"Loaded {MODEL_NAME} on {DEVICE}")

## 3. Inspect Model And Tokenizer Configuration

In [ ]:
print("Model type:", model.config.model_type)
print("Vocab size:", model.config.vocab_size)
print("Hidden size:", getattr(model.config, "hidden_size", None))
print("Layers:", getattr(model.config, "num_hidden_layers", None))
print("Attention heads:", getattr(model.config, "num_attention_heads", None))
print("Max position embeddings:", getattr(model.config, "max_position_embeddings", None))
print("Torch dtype:", getattr(model.config, "torch_dtype", None))

In [ ]:
print("EOS:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token, tokenizer.pad_token_id)
print("Special tokens:", tokenizer.special_tokens_map)

## 4. Tokenization

In [ ]:
examples = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "KV cache saves computation during decoding.",
    "def hello_world():\n    print('hello')",
]

for text in examples:
    token_ids = tokenizer.encode(text)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    print("TEXT:", repr(text))
    print("TOKEN COUNT:", len(token_ids))
    print("TOKEN IDS:", token_ids[:30])
    print("TOKENS:", tokens[:30])
    print("-" * 80)

## 5. Chat Template

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise teaching assistant."},
    {"role": "user", "content": "Explain KV cache in two sentences."},
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(formatted_prompt)
print("Token count:", len(tokenizer.encode(formatted_prompt)))

## 6. A Small Generation Helper

In [ ]:
def generate_reply(prompt, *, max_new_tokens=64, temperature=0.7, top_p=0.9, do_sample=True):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([text], return_tensors="pt").to(DEVICE)

    start = time.perf_counter()
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)
    elapsed = time.perf_counter() - start

    input_tokens = inputs["input_ids"].shape[-1]
    generated_ids = output[0][input_tokens:]
    generated_tokens = len(generated_ids)
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "prompt_tokens": input_tokens,
        "generated_tokens": generated_tokens,
        "elapsed_seconds": elapsed,
        "tokens_per_second": generated_tokens / elapsed if elapsed > 0 else None,
        "text": generated_text,
    }

In [ ]:
result = generate_reply("Explain what a tokenizer does in simple terms.", max_new_tokens=48, do_sample=True)

print("Prompt tokens:", result["prompt_tokens"])
print("Generated tokens:", result["generated_tokens"])
print("Elapsed seconds:", round(result["elapsed_seconds"], 2))
print("Tokens/sec:", round(result["tokens_per_second"], 2))
print("\nOutput:\n", result["text"])

## Practical task: Compare Decoding Settings

Try to change the decoding settings (temperature, top_p, sampling) and see how it affect generation.

- What changes when you change the decoding settings?
- Does this affect performance?
- Does this affect quality?


In [ ]:
# 1. Try with and without sampling.
# 2. Try with the different values of temperature (e.g. 1.0, 0.3)

## 7. Manual Prefill And Decode

In [ ]:
def tensor_nbytes(tensor):
    return tensor.numel() * tensor.element_size()

def past_key_values_nbytes(past_key_values):
    if hasattr(past_key_values, "layers"):
        return sum(
            tensor_nbytes(tensor)
            for layer in past_key_values.layers
            for tensor in layer.values
        )

    total = 0
    for layer in past_key_values:
        if isinstance(layer, (tuple, list)):
            total += sum(tensor_nbytes(tensor) for tensor in layer if torch.is_tensor(tensor))
        elif torch.is_tensor(layer):
            total += tensor_nbytes(layer)
    return total


def manual_prefill_decode(prompt, *, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([text], return_tensors="pt").to(DEVICE)
    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask")

    # Prefill: process the whole prompt and build the initial KV cache.
    prefill_start = time.perf_counter()
    with torch.inference_mode():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=True,
        )
    prefill_time = time.perf_counter() - prefill_start

    past_key_values = outputs.past_key_values
    next_token_logits = outputs.logits[:, -1, :]
    current_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
    generated_ids = [current_token_id.item()]

    if attention_mask is not None:
        attention_mask = torch.cat(
            [attention_mask, torch.ones((attention_mask.shape[0], 1), dtype=attention_mask.dtype, device=DEVICE)],
            dim=-1,
        )

    decode_step_times = []

    # Decode: process one generated token at a time and reuse the KV cache.
    for _ in range(max_new_tokens - 1):
        decode_start = time.perf_counter()
        with torch.inference_mode():
            outputs = model(
                input_ids=current_token_id,
                attention_mask=attention_mask,
                past_key_values=past_key_values,
                use_cache=True,
            )
        decode_step_times.append(time.perf_counter() - decode_start)

        past_key_values = outputs.past_key_values
        next_token_logits = outputs.logits[:, -1, :]
        current_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        token_id = current_token_id.item()
        generated_ids.append(token_id)

        if attention_mask is not None:
            attention_mask = torch.cat(
                [attention_mask, torch.ones((attention_mask.shape[0], 1), dtype=attention_mask.dtype, device=DEVICE)],
                dim=-1,
            )

        if token_id == tokenizer.eos_token_id:
            break

    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    total_decode_time = sum(decode_step_times)

    return {
        "prompt_tokens": input_ids.shape[-1],
        "generated_tokens": len(generated_ids),
        "prefill_time": prefill_time,
        "decode_step_times": decode_step_times,
        "total_decode_time": total_decode_time,
        "avg_decode_step_time": total_decode_time / len(decode_step_times) if decode_step_times else 0.0,
        "text": generated_text,
        "past_key_values_size_bytes": past_key_values_nbytes(past_key_values),
        "past_key_values_type": type(past_key_values).__name__,
    }

In [ ]:
result = manual_prefill_decode("Explain KV cache in one sentence.", max_new_tokens=16)

print("Prompt tokens:", result["prompt_tokens"])
print("Generated tokens:", result["generated_tokens"])
print("Prefill time:", round(result["prefill_time"], 4), "sec")
print("Total decode time:", round(result["total_decode_time"], 4), "sec")
print("Average decode step:", round(result["avg_decode_step_time"], 4), "sec")
print("KV cache size:", result["past_key_values_size"])
print("\nGenerated text:\n", result["text"])
print("\nFirst decode step times:", [round(t, 4) for t in result["decode_step_times"][:8]])

## Practical Task: Compare Prefill And Decode Under Different Conditions

Try short and long prompts with different output lengths. The exact numbers depend on hardware, but the distinction is what matters.

- How does changes to input / output affects performance?
- How does KV Cache size changes?

In [ ]:
# Try different size of input and output